## Convert Data to Embeddings

In [6]:
# cyberdata/notebook/create_email_embeddings.py

import json
import os
import pickle
import time
from pathlib import Path
from typing import Dict, List, Tuple

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

from cyberdata.utils.logger_config import setup_logger

# Set up logger
logger = setup_logger("cyberdata.scripts.create_email_embeddings")

# Load environment variables
load_dotenv()

# Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIMENSION = 3072  # Dimension for text-embedding-3-large
BATCH_SIZE = 50  # Reduced batch size to handle large emails better
RATE_LIMIT_DELAY = 1  # Seconds between API calls to avoid rate limiting

class EmailEmbeddingProcessor:
    """
    Processes email phishing dataset and creates FAISS embeddings for RAG usage.
    """
    
    def __init__(self, input_file: Path, output_dir: Path):
        """
        Initialize the processor.
        
        Args:
            input_file (Path): Path to the CSV file
            output_dir (Path): Directory to save FAISS index and metadata
        """
        self.input_file = input_file
        self.output_dir = output_dir
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        
        # Ensure output directory exists
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Initialize storage
        self.embeddings = []
        self.metadata = []
        self.processed_count = 0
        
        logger.info(f"Initialized EmailEmbeddingProcessor")
        logger.info(f"Input file: {self.input_file}")
        logger.info(f"Output directory: {self.output_dir}")
        logger.info(f"Using embedding model: {EMBEDDING_MODEL}")
    
    def load_dataset(self) -> pd.DataFrame:
        """
        Load the CSV dataset.
        
        Returns:
            pd.DataFrame: Loaded dataset
        """
        logger.info(f"Loading dataset from {self.input_file}")
        
        try:
            # Read the CSV file
            df = pd.read_csv(self.input_file, encoding='utf-8')
            
            logger.info(f"Successfully loaded dataset with {len(df)} rows")
            logger.info(f"Columns: {list(df.columns)}")
            
            # Display basic statistics
            if 'label' in df.columns:
                label_counts = df['label'].value_counts()
                logger.info(f"Label distribution: {label_counts.to_dict()}")
                
                # Calculate percentages
                total = len(df)
                scam_pct = (label_counts.get(1, 0) / total) * 100
                legit_pct = (label_counts.get(0, 0) / total) * 100
                logger.info(f"Scam emails: {scam_pct:.1f}%, Legitimate emails: {legit_pct:.1f}%")
            
            if 'source' in df.columns:
                source_counts = df['source'].value_counts()
                logger.info(f"Source distribution: {source_counts.to_dict()}")
            
            return df
            
        except Exception as e:
            logger.error(f"Error loading dataset: {str(e)}")
            raise
    
    def prepare_text_for_embedding(self, row: pd.Series) -> str:
        """
        Prepare email text for embedding by combining subject and body.
        
        Args:
            row (pd.Series): DataFrame row containing email data
            
        Returns:
            str: Combined text for embedding
        """
        subject = str(row.get('subject', '')).strip()
        body = str(row.get('body', '')).strip()
        
        # Handle missing values
        if subject in ['nan', 'None', '']:
            subject = '[No Subject]'
        if body in ['nan', 'None', '']:
            body = '[No Content]'
        
        # Combine subject and body with clear separation
        combined_text = f"Subject: {subject}\n\nBody: {body}"
        
        # Truncate based on token limits (8192 tokens max for text-embedding-3-large)
        # Conservative estimate: 1 token ≈ 3 characters for safety
        # Leave room for overhead: use 7000 tokens = ~21000 characters
        max_chars = 21000
        if len(combined_text) > max_chars:
            # Try to truncate intelligently - keep subject and start of body
            subject_part = f"Subject: {subject}\n\nBody: "
            remaining_chars = max_chars - len(subject_part) - 20  # 20 chars for truncation notice
            
            if remaining_chars > 100:  # Only truncate if we have reasonable space left
                truncated_body = body[:remaining_chars]
                combined_text = f"{subject_part}{truncated_body}...[TRUNCATED]"
            else:
                # If subject is too long, truncate everything more aggressively
                combined_text = combined_text[:max_chars] + "...[TRUNCATED]"
            
            logger.debug(f"Truncated long email text (original length: {len(f'Subject: {subject}\\n\\nBody: {body}')} -> {len(combined_text)})")
        
        return combined_text
    
    def create_metadata_record(self, row: pd.Series, index: int) -> Dict:
        """
        Create metadata record for an email.
        
        Args:
            row (pd.Series): DataFrame row containing email data
            index (int): Row index in the original dataset
            
        Returns:
            Dict: Metadata record
        """
        metadata = {
            'index': index,
            'label': int(row.get('label', -1)),  # -1 for unknown
            'source': str(row.get('source', 'unknown')),
            'subject': str(row.get('subject', '')),
            'body_length': len(str(row.get('body', ''))),
            'subject_length': len(str(row.get('subject', ''))),
            'is_scam': bool(row.get('label', 0) == 1),
            'is_legitimate': bool(row.get('label', 0) == 0)
        }
        
        return metadata
    
    def get_embeddings_batch(self, texts: List[str]) -> List[List[float]]:
        """
        Get embeddings for a batch of texts using OpenAI API.
        
        Args:
            texts (List[str]): List of texts to embed
            
        Returns:
            List[List[float]]: List of embedding vectors
        """
        try:
            logger.debug(f"Getting embeddings for batch of {len(texts)} texts")
            
            # Double-check text lengths before sending to API
            processed_texts = []
            for i, text in enumerate(texts):
                # Extra safety check: ensure no text is too long
                if len(text) > 21000:  # Conservative character limit
                    text = text[:21000] + "...[TRUNCATED]"
                    logger.warning(f"Text {i} was still too long, applying emergency truncation")
                processed_texts.append(text)
            
            response = self.client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=processed_texts
            )
            
            embeddings = [item.embedding for item in response.data]
            logger.debug(f"Successfully got {len(embeddings)} embeddings")
            
            return embeddings
            
        except Exception as e:
            logger.error(f"Error getting embeddings: {str(e)}")
            # Log the problematic texts for debugging
            for i, text in enumerate(texts):
                logger.debug(f"Text {i} length: {len(text)} characters")
                if len(text) > 21000:
                    logger.warning(f"Text {i} exceeds safe length: {len(text)} characters")
            raise
    
    def process_dataset(self, df: pd.DataFrame, sample_size: int = None) -> Tuple[np.ndarray, List[Dict]]:
        """
        Process the entire dataset to create embeddings and metadata.
        
        Args:
            df (pd.DataFrame): Dataset to process
            sample_size (int, optional): If provided, only process a sample of this size
            
        Returns:
            Tuple[np.ndarray, List[Dict]]: Embeddings array and metadata list
        """
        if sample_size and sample_size < len(df):
            logger.info(f"Sampling {sample_size} rows from dataset of {len(df)} rows")
            df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        
        total_rows = len(df)
        logger.info(f"Processing {total_rows} emails for embedding")
        
        all_embeddings = []
        all_metadata = []
        
        # Process in batches
        for i in tqdm(range(0, total_rows, BATCH_SIZE), desc="Creating embeddings"):
            batch_end = min(i + BATCH_SIZE, total_rows)
            batch_df = df.iloc[i:batch_end]
            
            # Prepare texts for this batch
            batch_texts = []
            batch_metadata = []
            
            for idx, row in batch_df.iterrows():
                text = self.prepare_text_for_embedding(row)
                metadata = self.create_metadata_record(row, idx)
                
                batch_texts.append(text)
                batch_metadata.append(metadata)
            
            # Get embeddings for this batch
            try:
                batch_embeddings = self.get_embeddings_batch(batch_texts)
                
                # Store results
                all_embeddings.extend(batch_embeddings)
                all_metadata.extend(batch_metadata)
                
                self.processed_count += len(batch_embeddings)
                
                # Rate limiting
                if i + BATCH_SIZE < total_rows:  # Don't sleep after the last batch
                    time.sleep(RATE_LIMIT_DELAY)
                
            except Exception as e:
                logger.error(f"Error processing batch {i}-{batch_end}: {str(e)}")
                # Continue with next batch instead of failing completely
                continue
        
        logger.info(f"Successfully processed {self.processed_count} emails")
        
        # Convert to numpy array
        embeddings_array = np.array(all_embeddings, dtype=np.float32)
        logger.info(f"Created embeddings array with shape: {embeddings_array.shape}")
        
        return embeddings_array, all_metadata
    
    def create_faiss_index(self, embeddings: np.ndarray) -> faiss.Index:
        """
        Create a FAISS index for efficient similarity search.
        
        Args:
            embeddings (np.ndarray): Embeddings array
            
        Returns:
            faiss.Index: FAISS index
        """
        logger.info("Creating FAISS index")
        
        # Use IndexFlatIP for cosine similarity (inner product with normalized vectors)
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        
        # Create index
        index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
        
        # Add embeddings to index
        index.add(embeddings)
        
        logger.info(f"FAISS index created with {index.ntotal} vectors")
        return index
    
    def save_index_and_metadata(self, index: faiss.Index, metadata: List[Dict]):
        """
        Save FAISS index and metadata to disk.
        
        Args:
            index (faiss.Index): FAISS index to save
            metadata (List[Dict]): Metadata to save
        """
        logger.info("Saving FAISS index and metadata")
        
        # Save FAISS index
        index_path = self.output_dir / "faiss_index.bin"
        faiss.write_index(index, str(index_path))
        logger.info(f"FAISS index saved to: {index_path}")
        
        # Save metadata as JSON
        metadata_path = self.output_dir / "metadata.json"
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        logger.info(f"Metadata saved to: {metadata_path}")
        
        # Save metadata as pickle for faster loading
        metadata_pickle_path = self.output_dir / "metadata.pkl"
        with open(metadata_pickle_path, 'wb') as f:
            pickle.dump(metadata, f)
        logger.info(f"Metadata pickle saved to: {metadata_pickle_path}")
        
        # Save configuration and statistics
        config = {
            'embedding_model': EMBEDDING_MODEL,
            'embedding_dimension': EMBEDDING_DIMENSION,
            'total_vectors': index.ntotal,
            'batch_size': BATCH_SIZE,
            'processed_count': self.processed_count,
            'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            'input_file': str(self.input_file),
            'output_dir': str(self.output_dir)
        }
        
        config_path = self.output_dir / "config.json"
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=2)
        logger.info(f"Configuration saved to: {config_path}")
        
        # Create summary statistics
        if metadata:
            stats = self.calculate_statistics(metadata)
            stats_path = self.output_dir / "statistics.json"
            with open(stats_path, 'w', encoding='utf-8') as f:
                json.dump(stats, f, indent=2)
            logger.info(f"Statistics saved to: {stats_path}")
    
    def calculate_statistics(self, metadata: List[Dict]) -> Dict:
        """
        Calculate statistics about the embedded dataset.
        
        Args:
            metadata (List[Dict]): Metadata list
            
        Returns:
            Dict: Statistics
        """
        if not metadata:
            return {}
        
        total = len(metadata)
        scam_count = sum(1 for m in metadata if m.get('is_scam', False))
        legit_count = sum(1 for m in metadata if m.get('is_legitimate', False))
        
        sources = [m.get('source', 'unknown') for m in metadata]
        source_counts = {source: sources.count(source) for source in set(sources)}
        
        body_lengths = [m.get('body_length', 0) for m in metadata]
        subject_lengths = [m.get('subject_length', 0) for m in metadata]
        
        stats = {
            'total_emails': total,
            'scam_emails': scam_count,
            'legitimate_emails': legit_count,
            'scam_percentage': (scam_count / total * 100) if total > 0 else 0,
            'source_distribution': source_counts,
            'average_body_length': sum(body_lengths) / len(body_lengths) if body_lengths else 0,
            'average_subject_length': sum(subject_lengths) / len(subject_lengths) if subject_lengths else 0
        }
        
        return stats
    
    def process_complete_pipeline(self, sample_size: int = None):
        """
        Run the complete pipeline: load data, create embeddings, build index, save results.
        
        Args:
            sample_size (int, optional): If provided, only process a sample
        """
        logger.info("Starting complete embedding pipeline")
        
        try:
            # Step 1: Load dataset
            df = self.load_dataset()
            
            # Step 2: Process dataset to create embeddings
            embeddings, metadata = self.process_dataset(df, sample_size)
            
            # Step 3: Create FAISS index
            index = self.create_faiss_index(embeddings)
            
            # Step 4: Save everything
            self.save_index_and_metadata(index, metadata)
            
            logger.info("Pipeline completed successfully!")
            logger.info(f"Created embeddings for {len(metadata)} emails")
            logger.info(f"Files saved in: {self.output_dir}")
            
        except Exception as e:
            logger.error(f"Pipeline failed: {str(e)}", exc_info=True)
            raise


def main():
    """
    Main function to run the embedding creation pipeline.
    """
    # Define paths for notebook environment
    # Assuming script is run from PROJECT_ROOT/notebook/
    current_dir = Path.cwd()
    
    # Check if we're in the notebook directory
    if current_dir.name == 'notebook':
        project_root = current_dir.parent
    else:
        # Try to find project root by looking for specific files
        project_root = current_dir
        while project_root.parent != project_root:
            if (project_root / "pyproject.toml").exists() or (project_root / "raw").exists():
                break
            project_root = project_root.parent
        else:
            # Fallback: assume current directory is project root
            project_root = current_dir
    
    input_file = project_root / "raw" / "five_email_phishing.csv"
    output_dir = project_root / "db" / "five_email_phishing"
    
    logger.info("Email Phishing Dataset Embedding Pipeline")
    logger.info(f"Project root: {project_root}")
    logger.info(f"Input file: {input_file}")
    logger.info(f"Output directory: {output_dir}")
    
    # Check if input file exists
    if not input_file.exists():
        logger.error(f"Input file not found: {input_file}")
        print(f"Error: Input file not found: {input_file}")
        print(f"Please ensure the file exists at: {input_file}")
        return
    
    # Check for OpenAI API key
    if not os.getenv("OPENAI_API_KEY"):
        logger.error("OpenAI API key not found in environment variables")
        print("Error: Please set OPENAI_API_KEY environment variable")
        print("You can set it in your notebook with:")
        print("import os")
        print("os.environ['OPENAI_API_KEY'] = 'your-api-key-here'")
        return
    
    # Initialize processor
    processor = EmailEmbeddingProcessor(input_file, output_dir)
    
    # Run the pipeline
    # For testing, you can add sample_size parameter: processor.process_complete_pipeline(sample_size=1000)
    processor.process_complete_pipeline()
    
    print("\n" + "="*60)
    print("EMBEDDING CREATION COMPLETED")
    print("="*60)
    print(f"Input file: {input_file}")
    print(f"Output directory: {output_dir}")
    print(f"FAISS index: {output_dir / 'faiss_index.bin'}")
    print(f"Metadata: {output_dir / 'metadata.json'}")
    print(f"Configuration: {output_dir / 'config.json'}")
    print(f"Statistics: {output_dir / 'statistics.json'}")
    print("="*60)


if __name__ == '__main__':
    main()

2025-06-08 20:33:32,200 - cyberdata.scripts.create_email_embeddings - INFO - C:\Users\Tianyu Wang\Documents\Notebooks\cyberdata\src\cyberdata\utils\logger_config.py:62 - setup_logger() - Logger initialized. Logs will be saved to C:\Users\Tianyu Wang\Documents\Notebooks\cyberdata\logs\cyberdata.scripts.create_email_embeddings_20250608_203332.log
2025-06-08 20:33:32,204 - cyberdata.scripts.create_email_embeddings - INFO - C:\Users\Tianyu Wang\AppData\Local\Temp\ipykernel_20212\1613244265.py:437 - main() - Email Phishing Dataset Embedding Pipeline
2025-06-08 20:33:32,205 - cyberdata.scripts.create_email_embeddings - INFO - C:\Users\Tianyu Wang\AppData\Local\Temp\ipykernel_20212\1613244265.py:438 - main() - Project root: c:\Users\Tianyu Wang\Documents\Notebooks\cyberdata
2025-06-08 20:33:32,206 - cyberdata.scripts.create_email_embeddings - INFO - C:\Users\Tianyu Wang\AppData\Local\Temp\ipykernel_20212\1613244265.py:439 - main() - Input file: c:\Users\Tianyu Wang\Documents\Notebooks\cyberda

KeyboardInterrupt: 